<a href="https://colab.research.google.com/github/nayakamhrudaya/GenAI/blob/main/VideoToSummary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U gradio openai-whisper ffmpeg-python openai
!apt-get install -y ffmpeg
import gradio as gr
import whisper
import ffmpeg
import os
from openai import OpenAI

# Load Whisper model
whisper_model = whisper.load_model("base")

AUDIO_FILE = "audio.wav"

# -----------------------------
# Extract audio from uploaded video
# -----------------------------
def extract_audio(video_path):
    stream = ffmpeg.input(video_path)
    stream = ffmpeg.output(stream, AUDIO_FILE, format='wav')
    ffmpeg.run(stream, overwrite_output=True, quiet=True)
    return AUDIO_FILE

# -----------------------------
# Transcribe + Translate (ANY LANGUAGE → ENGLISH)
# -----------------------------
def transcribe(audio_path):
    result = whisper_model.transcribe(audio_path, task="translate")
    return result["text"]

# -----------------------------
# Summarize using OpenAI
# -----------------------------
def summarize(api_key, text, lang):
    client = OpenAI(api_key=api_key)

    if not lang:
        lang = "English"

    prompt = f"""
    Summarize the following transcript in {lang}.
    Make it clear, structured, and concise:

    {text}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    return response.choices[0].message.content

# -----------------------------
# MAIN PIPELINE
# -----------------------------
def process(api_key, file, lang):
    try:
        if not api_key:
            return " Please enter OpenAI API key"

        if file is None:
            return " Please upload a video file"

        # Step 1: audio extraction
        audio_path = extract_audio(file)

        # Step 2: transcription + translation
        text = transcribe(audio_path)

        # Step 3: summarization
        summary = summarize(api_key, text, lang)

        # cleanup
        if os.path.exists(audio_path):
            os.remove(audio_path)

        return summary

    except Exception as e:
        return f" ERROR: {str(e)}"

# -----------------------------
# UI (UPLOAD ONLY)
# -----------------------------
app = gr.Interface(
    fn=process,
    inputs=[
        gr.Textbox(label="OpenAI API Key", type="password"),
        gr.Video(label="Upload Video File"),
        gr.Dropdown(
            choices=["English", "Hindi", "Spanish", "French", "German", "Japanese","Telugu"],
            value="English",
            label="Summary Language (default English)"
        )
    ],
    outputs="text",
    title="AI Video Summarizer (Upload Only - Stable)",
    description="Upload video → auto transcription → translation → AI summary",
    flagging_mode="never"
)

app.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 8.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.3 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=9ee8968fcf8ed0843f6b523d8ae7af69d8b7be77c3a009ef447d910fb25ec6a9
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing ins

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 121MiB/s]


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a5576d2627fb5ffa0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
